# DSS — you get what you ask for

DSS finds spatial filters that maximise a property you *declare*, rather than variance. Declaring the right property is the whole job.

*Deep dive behind the [five-minute demo](../meta_mne_denoise_demo.ipynb).*

## Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("demo_utils.py").exists():
    done = subprocess.run(
        ["git", "clone", "-q", "--depth", "1",
         "https://github.com/snesmaeili/mne-denoise-meta-demo.git"],
        capture_output=True, text=True)
    if done.returncode != 0:
        raise SystemExit(
            "Could not clone the demo repository. If it is still private, the Colab "
            "VM has no credentials for it -- authorising Colab lets it OPEN a "
            "notebook, not clone the repo. Make it public, or run locally."
        )
    os.chdir("mne-denoise-meta-demo")
    sys.path.insert(0, os.getcwd())
try:
    import mne_denoise  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mne-denoise @ git+https://github.com/mne-tools/mne-denoise.git@f5b821cc2a535e84ed46085d45ea5a356dd8d548"],
                   check=True)

import warnings, logging
import numpy as np
import matplotlib.pyplot as plt
import mne

mne.set_log_level("ERROR")
logging.getLogger("mne_denoise").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*Epochs are not baseline corrected.*")
%matplotlib inline
RANDOM_STATE = 97

## Same data, two different biases

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_ep, n_ch, n_t, sfreq = 120, 32, 256, 256.0
times = np.arange(n_t) / sfreq - 0.2

# an evoked response that repeats across trials
evoked = -np.exp(-((times - 0.15) ** 2) / 0.002)
v_evoked = rng.standard_normal(n_ch); v_evoked /= np.linalg.norm(v_evoked)

# a 10 Hz rhythm with random phase per trial (NOT phase-locked)
v_alpha = rng.standard_normal(n_ch); v_alpha /= np.linalg.norm(v_alpha)

data = rng.standard_normal((n_ep, n_ch, n_t)) * 0.7
for e in range(n_ep):
    data[e] += np.outer(v_evoked, evoked) * 1.1
    data[e] += np.outer(v_alpha, np.sin(2 * np.pi * 10 * times + rng.uniform(0, 6.28))) * 1.4

info = mne.create_info(n_ch, sfreq, "eeg")
epochs = mne.EpochsArray(data, info, tmin=-0.2, verbose="ERROR")
print(f"{n_ep} trials: a phase-locked evoked response AND a stronger non-phase-locked 10 Hz rhythm")

In [ ]:
from mne_denoise.dss import DSS, AverageBias, BandpassBias

dss_avg = DSS(bias=AverageBias(axis="epochs"), n_components=4)
src_avg = dss_avg.fit_transform(epochs)

dss_band = DSS(bias=BandpassBias(freq_band=(8.0, 12.0), sfreq=sfreq), n_components=4)
src_band = dss_band.fit_transform(epochs)

# PCA is run, not assumed. It maximises variance, and the alpha was planted
# stronger than the evoked -- so this is a prediction the cell has to survive.
flat = data.transpose(1, 0, 2).reshape(n_ch, -1)
u, s, _ = np.linalg.svd(flat - flat.mean(axis=1, keepdims=True), full_matrices=False)

print("AverageBias  eigenvalues:", np.round(dss_avg.eigenvalues_[:4], 3))
print("BandpassBias eigenvalues:", np.round(dss_band.eigenvalues_[:4], 3))
print("PCA          explained variance ratio:", np.round(s[:4] ** 2 / np.sum(s ** 2), 3))
print("\n|cos| of component 1 against each planted pattern:")
for label, pattern in [("PCA", u[:, 0]),
                       ("AverageBias", dss_avg.patterns_[:, 0]),
                       ("BandpassBias", dss_band.patterns_[:, 0])]:
    p = pattern / np.linalg.norm(pattern)
    print(f"  {label:14s} evoked {abs(p @ v_evoked):.3f}   alpha {abs(p @ v_alpha):.3f}")

The alpha rhythm carries more variance, so PCA returns it — measured above, not assumed. `AverageBias` returns the evoked response from the same data, and `BandpassBias` returns the rhythm. One estimator, one argument changed, three different answers.

This is the part that matters: PCA has no way to be *asked* for the evoked source. Neither does Xdawn, or SSD, or CSP. Each of them is this same generalised eigenvalue problem with the criterion welded shut.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for ax, src, title, colour in [
    (axes[0], src_avg, "AverageBias — component 1", "#0072B2"),
    (axes[1], src_band, "BandpassBias 8-12 Hz — component 1", "#009E73"),
]:
    for tr in np.asarray(src)[::4, 0, :]:
        ax.plot(times, tr, color=colour, alpha=0.06, lw=0.8)
    ax.plot(times, np.asarray(src)[:, 0, :].mean(0), color=colour, lw=3)
    ax.axvline(0, color="#888888", ls=":", lw=1)
    ax.set_title(title); ax.set_xlabel("Time (s)")
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

## The bias carries all of the content

If the bias is the identity — "everything is equally interesting" — every direction scores the same and DSS has nothing to say. That is the degenerate case the whole method sits on top of.

In [ ]:
dss_id = DSS(bias=lambda x: x, n_components=8)   # identity: declare nothing
dss_id.fit(data.transpose(1, 2, 0))              # (n_channels, n_times, n_epochs)

print("identity bias, eigenvalues:", np.round(dss_id.eigenvalues_[:8], 6))
print(f"max deviation from 1.0: {np.abs(dss_id.eigenvalues_ - 1.0).max():.2e}")
print("\nEvery direction scores identically. Declare nothing, learn nothing --")
print("the bias is not a tuning knob on top of DSS, it is the entire content.")

## Linear vs non-linear — and where ICA turns up

Everything above is a **linear** bias: a fixed operator applied to the data, so the solution is a closed-form generalised eigendecomposition. That is the same class as Xdawn, SSD and CSP.

A **non-linear** denoiser is different in kind. It operates on the current source estimate and is re-estimated at every iteration, so there is no closed form — `IterativeDSS` runs a fixed-point loop instead. `mne_denoise/dss/nonlinear.py` claims this makes DSS *"equivalent to FastICA when using ICA contrast functions (tanh, gauss, cube)"*.

That is a testable claim, so here it is tested against `sklearn`'s FastICA rather than repeated.

In [ ]:
from scipy import stats
from sklearn.decomposition import FastICA

from mne_denoise.dss import IterativeDSS, TanhMaskDenoiser, beta_tanh

rng2 = np.random.default_rng(RANDOM_STATE)
n = 4000
t = np.linspace(0, 8, n)
S = np.vstack([
    stats.laplace.rvs(size=n, random_state=1),   # super-Gaussian, sparse
    np.sign(np.sin(3 * t)),                      # square wave, high kurtosis
    np.sin(10 * t),                              # sub-Gaussian
    rng2.standard_normal(n),                     # Gaussian -- not recoverable in principle
])
S /= S.std(axis=1, keepdims=True)
X = rng2.standard_normal((8, 4)) @ S

S_dss = IterativeDSS(TanhMaskDenoiser(), n_components=4,
                     beta=beta_tanh, random_state=0).fit_transform(X)
S_ica = FastICA(n_components=4, fun="logcosh", random_state=0,
                max_iter=1000, whiten="unit-variance").fit_transform(X.T).T

def recovery(rec):
    c = np.abs(np.corrcoef(rec, S)[: rec.shape[0], rec.shape[0]:])
    return c.max(axis=0)

labels = ["laplace", "square", "sinusoid", "gaussian"]
print(f"{'source':<12s} {'IterativeDSS':>13s} {'FastICA':>9s}")
for name, a, b in zip(labels, recovery(S_dss), recovery(S_ica)):
    print(f"{name:<12s} {a:>13.3f} {b:>9.3f}")

pair = np.abs(np.corrcoef(S_dss, S_ica)[:4, 4:]).max(axis=1)
print(f"\nDSS component vs its best-matching FastICA component: {np.round(pair, 3).tolist()}")
print("Two land on top of FastICA; two agree to about 0.89. Close, not identical --")
print("and FastICA recovered all four here, so this is not a win, it is a family resemblance.")

> **And the caution.** On real ERP CORE N170 data, trial-average DSS improved split-half reproducibility in 37 of 40 participants but held-out faces-vs-cars discriminability in only 25 of 40. Concentrating what repeats across trials is not the same as sharpening a condition contrast.

> **And the honest comparison.** On that same task DSS beat matched-rank PCA in 32 of 40 participants, but beat `mne.decoding.XdawnTransformer` in only **15 of 40** — Xdawn is better on the median. Plain PCA beat raw sensor space in **40 of 40**, which is why "DSS improves on doing nothing" is a much weaker claim than it sounds.
>
> Two caveats that cut against reading too much into that: Xdawn's regularisation was **not tuned** (fixed at `ledoit_wolf`), so its occasional bad splits are reported as observed rather than as a property of Xdawn; and the condition-AUC arms differ in dimensionality (DSS ≈ 7.7 components vs 30 sensor channels), so part of that comparison is regularisation rather than filtering.
>
> None of that touches the argument this notebook actually makes. DSS's case is not that it wins the evoked-enhancement benchmark — it doesn't. It is that the criterion is an argument, and it does not have to be a covariance.